# TriFuseSite quick start
Synthetic forward-pass example with the three intermediate representations.


In [ ]:
import numpy as np
import torch
from trifusesite import TriFuseSite
from trifusesite.data.features import handcrafted62
from trifusesite.data.graphs import radius_graph
torch.manual_seed(17)
sequence = 'ACDEFGHIK'
n = len(sequence)
# All geometry is missing in this artificial example.
ca = np.zeros((n, 3), np.float32)
mask = np.zeros(n, bool)
adj, edges, edge_features, fallback = radius_graph(ca, mask)
inputs = dict(residue_features=torch.tensor(handcrafted62(sequence)),
              esm_embedding=torch.zeros(n, 1280),
              patch_adjacency=torch.tensor(adj),
              global_edge_index=torch.tensor(edges),
              global_edge_features=torch.tensor(edge_features),
              ca=torch.tensor(ca), structure_mask=torch.tensor(mask),
              confidence=torch.zeros(n))


In [ ]:
model = TriFuseSite(task='smpi').eval()
with torch.no_grad():
    logits, states = model(**inputs, return_intermediates=True)
assert logits.shape == (n, 2)
assert states['geometry_gate'].item() == 0
print({k: tuple(states[k].shape) for k in ('X0', 'Hgraph', 'H')})


For prediction with trained weights, use `main_predict.py` with a task checkpoint and prepared input manifest.
